# Week 3: Differential Gene Expression

Differential gene expression (DGE) analysis explores the differences in gene expression between two separate samples. This can be achieved through performing bulk RNA sequencing (bulk RNA-seq) on various samples of interest and corresponding controls.

In this module, we will focus on the computational analysis of bulk RNA-Seq data. We'll explore the use of Python-based tools for this analysis, including understanding function parameters and interpreting the results.

## Learning objectives
By the end of this module, you will be equipped to accomplish the following tasks:

- Apply necessary preprocessing steps to the tabular data
- Perform exploratory data analysis to look at data quality
- Conduct differential expression analysis
- Interpret the results of the identified differentially expressed genes

Let's start by installing the required libraries and importing them.

In [ ]:
!pip -q install pyDESeq2 mygene

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns

from mygene import MyGeneInfo

import scipy.stats as stats
from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats

In [ ]:
# This cell contains helper functions we'll use in this module.
# You don't need to understand this code, but make sure to run the cell!
def plot_volcano(dge_df, p_threshold=0.05, log2fc_threshold=1, top_N=10):
    dge_df = dge_df.reset_index()
    plt.figure(figsize=(12, 8))

    # Plot non-significant genes in gray
    plt.scatter(dge_df['log2FoldChange'], -np.log10(dge_df['padj']), edgecolors='lightgray', facecolors='none', label='Non-significant', alpha=0.7)

    # Plot upregulated genes in organge
    plt.scatter(dge_df[(dge_df['log2FoldChange'] > log2fc_threshold) & (dge_df['padj'] < p_threshold)]['log2FoldChange'],
                -np.log10(dge_df[(dge_df['log2FoldChange'] > log2fc_threshold) & (dge_df['padj'] < p_threshold)]['padj']),
                edgecolors='orange', facecolors='none', label='Upregulated', alpha=0.7)
    
    # Plot downregulated genes in blue
    plt.scatter(dge_df[(dge_df['log2FoldChange'] < -log2fc_threshold) & (dge_df['padj'] < p_threshold)]['log2FoldChange'],
                -np.log10(dge_df[(dge_df['log2FoldChange'] < -log2fc_threshold) & (dge_df['padj'] < p_threshold)]['padj']),
                edgecolors='skyblue', facecolors='none', label='Downregulated', alpha=0.7)

    # Plotting and labeling most significant genes with fill
    if top_N > 0:
        top_genes = dge_df.sort_values('padj', ascending=True).head(top_N)
        for _, row in top_genes.iterrows():
            color = 'orange' if row['log2FoldChange'] > 0 else 'skyblue'
            plt.scatter(row['log2FoldChange'], -np.log10(row['padj']), edgecolors=color, facecolors=color, s=50)
            plt.text(row['log2FoldChange'], -np.log10(row['padj']), row['symbol'], fontsize=9, ha='right')

    # Adding lines for thresholds
    sig_line = -np.log10(p_threshold)
    plt.axhline(y=sig_line, color='black', linestyle='--', linewidth=0.5)
    plt.axvline(x=log2fc_threshold, color='black', linestyle='--', linewidth=0.5)
    plt.axvline(x=-log2fc_threshold, color='black', linestyle='--', linewidth=0.5)

    # Adding details to plot
    plt.title('Volcano Plot of Differential Gene Expression')
    plt.xlabel('Log2 Fold Change')
    plt.ylabel('-Log10 Adjusted P-Value')
    plt.legend()
    plt.show()

# This is a helper function
def plot_heatmap(
    gene_counts, 
    metadata, 
    title_prefix, 
    condition_col='treatment_protocol',
    control_name='Untreated',
    xticklabels=True
):
    # Z-score normalization 
    counts_z = gene_counts.apply(stats.zscore, axis=0)

    # Map the control to blue and all other conditions to red
    unique_conditions = metadata[condition_col].unique()
    color_map = {}
    for condition in unique_conditions:
        if condition == control_name:
            color_map[condition] = 'blue'
        else:
            color_map[condition] = 'red'
            
    row_colors = metadata[condition_col].map(color_map)

    # Create the clustermap
    print(f"Plotting heatmap for: {title_prefix}")
    print("Condition colors:", color_map)

    sns.clustermap(
        counts_z,
        cmap = mcolors.LinearSegmentedColormap.from_list("blue_white_orange", ['#2eb2e8', "white", 'orange'], N=256),
        row_colors=row_colors,
        col_cluster=True,
        row_cluster=True,
        linewidths=0.5,
        linecolor='gray',
        xticklabels=xticklabels,
        figsize=(12, 8)
    )

    # Adding details to plot
    plt.suptitle(f'{title_prefix} - Expression Heatmap', y=1.02, fontsize=16)
    plt.show()


def extract_top_bottom_genes(genes_counts, dge_df, N=10, min_counts=10):
    """
    Extracts the top and bottom N genes based on -log10(p-values) and returns their count data.

    Parameters:
        genes_counts (pd.DataFrame): Gene expression count matrix (samples x genes).
        dge_df (pd.DataFrame): DataFrame with DGE results, including 'padj' and 'symbol' columns.
        N (int): Number of top and bottom genes to extract.
        min_counts (int): Minimum number of gene counts considered during data cleaning.

    Returns:
        top_genes_counts (pd.DataFrame): Counts of the top N genes.
        bot_genes_counts (pd.DataFrame): Counts of the bottom N genes.
        top_genes (list): List of top N gene names.
        bot_genes (list): List of bottom N gene names.
    """
    # Create DataFrame
    df = pd.DataFrame({
        'gene_counts': genes_counts.sum(axis=0),
        'log_p_value': -np.log10(dge_df['padj'])
    })
    df = df.reset_index()
    
    # Drop genes (col) that are below threshold counts
    df = df.loc[df['gene_counts'] > min_counts]

    # Drop genes (col) that are NaN or Inf
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df.dropna(axis=0)

    df_sorted = df.sort_values(by='log_p_value', ascending=False)

    # Get top and bottom N genes
    top_genes = df_sorted.head(N)['symbol'].tolist()
    bot_genes = df_sorted.tail(N)['symbol'].tolist()

    # Map top and bottom N genes to log_p_values
    top_gene_counts = genes_counts.loc[:, top_genes]
    bot_gene_counts = genes_counts.loc[:, bot_genes]

    return top_gene_counts, bot_gene_counts, top_genes, bot_genes

## Analyzing Bulk RNA-seq Data

### Transcriptomics Data

RNA-seq data comes in two flavours: 
1. **Bulk RNA-seq** analyzes RNA from a population of cells (e.g., a tissue sample). This gives an average gene expression profile, which is useful for comparing overall states between samples. This will be the focus of this week and next week's tutorials.
2. **Single-Cell RNA-seq (scRNA-seq)** measures gene expression in thousands of individual cells simultaneously. This powerful technique reveals cell-to-cell variation that is missed in bulk analysis. We'll be exploring this in later tutorials.

By analyzing bulk RNA-seq data, we seek to answer questions about:
- **Differential Expression:** Which genes are more or less active between different groups (e.g., treated vs. untreated)?
- **Biological Pathways:** What are the functions of the genes that show changing expression levels?
- **Cellular States:** How do gene expression profiles define a specific cell type or disease state?
- **Gene Regulation:** What mechanisms might be controlling these changes in gene expression?


### Our Dataset
We will work with a bulk RNA-Seq dataset from Himes et al. (2014) that investigated how the asthma treatments `'Dexamethasone'` and `'Albuterol'` affect gene expression in human airway smooth muscle cells.

The data is provided in `airway_rawcounts_more_exps.csv` as a count matrix, with genes as rows (Ensembl IDs) and samples as columns (SRR IDs). For this tutorial, we will only use the `'Untreated'` and `'Dexamethasone'` samples as control and treatment groups, respectively.

**Source:** *Himes BE, Jiang X, Wagner P, Hu R, Wang Q, Klanderman B, Whitaker RM, Duan Q, Lasky-Su J, Nikolos C, Jester W, Johnson M, Panettieri R Jr, Tantisira KG, Weiss ST, Lu Q. “RNA-Seq Transcriptome Profiling Identifies CRISPLD2 as a Glucocorticoid Responsive Gene that Modulates Cytokine Function in Airway Smooth Muscle Cells.” PLoS One. 2014 Jun 13;9(6):e99625. PMID: 24926665. GEO: GSE52778.*

In [ ]:
# Load in raw counts dataset
counts_df = pd.read_csv('data/airway_rawcounts_more_exps.csv')
counts_df

Upon first inspection, we can see the `ensgene` column, which represents the Ensembl gene IDs that were investigated during the study.

Subsequent columns are labeled `SRR#######` and correspond to individual samples of including control and treatment groups, which we'll learn more about when we access the metadata for this dataset.

For now, we'll need to clean up this data by:
1. Removing all the genes that don't have any counts.

In [ ]:
# Select only columns with numeric expression count values
numeric_cols = counts_df.select_dtypes(include=np.number).columns
print(f"{counts_df.shape[0]} genes investigated in original study")

# Calculate the sum of counts for each gene across all samples
row_sums = counts_df[numeric_cols].sum(axis=1)

# Filter the DataFrame to retain only rows with nonzero total counts
counts_df = counts_df[row_sums > 0]
print(f"{counts_df.shape[0]} genes with non-zero expression counts")

2. Mapping Ensembl gene IDs to more useful symbolic gene IDs.

The Ensembl gene IDs (e.g., `ENSG00000223972`) are unique and stable identifiers, but they are not very human-readable. To make our results easier to interpret, we will convert them to common gene symbols (e.g., `DDX11L1`).

We will use the `mygene` Python library, which provides a simple way to query the [MyGene.info](https://mygene.info/) database, a service that aggregates gene annotation data from many sources.

In [ ]:
# Get list of all Ensembl gene IDs
ensembl_ids = counts_df['ensgene'].tolist()

# Query mygene API to convert to symbolic gene IDs
results = MyGeneInfo().querymany(
    ensembl_ids,
    scopes='ensembl.gene',
    fields='symbol',
    species='human',
    as_dataframe=True,
    df_index=True
)

# Map Symbolic gene IDs to Ensembl gene IDs
mapping = results['symbol'].to_dict()
counts_df['symbol'] = counts_df['ensgene'].map(mapping) # Apply mapping
sym_counts_df = counts_df.drop(columns=['ensgene'])     # Remove Ensembl gene ID column
sym_counts_df

3. Remove any IDs that were duplicates and could not be mapped. Follow the following exercises:


---
**Q*1: Remove NaN rows.**

> Hint: Refer to the documentation for `DataFrame.dropna()` [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.dropna.html) and set `subset=['symbol']`.

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# YOUR CODE HERE
sym_counts_df = ...  # Remove unfound genes

---
**Q*2: Remove duplicate rows.**

> Hint: Refer to the documentation for `DataFrame.drop_duplicates` [here](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html) and set `subset='symbol'` and `keep='first'`.

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# YOUR CODE HERE
sym_counts_df = ...  # Remove duplicate genes

---

In [ ]:
# Clean up dataset
sym_counts_df = sym_counts_df.set_index('symbol')       # Set index to symbolic gene ID

print(f"{sym_counts_df.shape[0]} non-duplicate genes with symbolic IDs")

# Transpose the DataFrame. The reason we do this is because the pyDESeq2 library expects the data to be in the format of (samples x genes),
# and our current DataFrame is in the format (genes x samples).
sym_counts_df = sym_counts_df.T
sym_counts_df

With our data setup in the right format, we now need to map our SRR IDs to the experimental conditions using the metadata file.

In [ ]:
# Load in metadata
metadata_df = pd.read_csv('data/airway_metadata_more_exps.csv', index_col='srr_id')
metadata_df

By inspecting this relatively small DataFrame, we can identify the treatment applied and the specific airway smooth muscle cell lines examined in this study. The other columns represent alternative IDs for these samples.

We will compare two treatment conditions: the `'Untreated'` samples, which serve as the control, and the `'Dexamethosone'` treated samples as a test treatment to examine the relative differences in expression profiles between these two conditions.

In [ ]:
# First, let's keep the metadata only for the Untreated and Dexamethosone treatment samples
dex_vs_con_metadata_df = metadata_df.loc[(metadata_df['treatment_protocol'] == 'Dexamethasone') | (metadata_df['treatment_protocol'] == 'Untreated')]
dex_vs_con_metadata_df

In [ ]:
# Now, let's extract the corresponding samples from sym_counts_df

# Dexamethosone treatment group
group_dex = sym_counts_df.loc[metadata_df['treatment_protocol'] == 'Dexamethasone', :]
print(f"Dexamathosone treated samples: {group_dex.index.tolist()}")

# Untreated control group
group_con = sym_counts_df.loc[metadata_df['treatment_protocol'] == 'Untreated', :]
print(f"Untreated control samples: {group_con.index.tolist()}")

# Combined Dexamethosone treatment and control groups
dex_vs_con_counts_df = sym_counts_df.loc[(metadata_df['treatment_protocol'] == 'Dexamethasone') | (metadata_df['treatment_protocol'] == 'Untreated'), :]
dex_vs_con_counts_df

## Wald Test using `PyDESeq2`

With our dataset, we now have to consider how we'll compare our control to our test group. We can't perform a simple t-test because our RNA-seq count data violates its core assumptions:
1. **The count data uses discrete integers (e.g., 0, 1, 2, 7, 100), not continuous values (e.g., 0.1, 6.3, 20.0, 100.006).**
    - Our count data follows a Negative Binomial distribution, which is highly skewed.
    - A t-test requires a Normal distribution with an even distribution of variance.
2. **The variance is linked to the mean.**
    - Our count data can have *low average gene counts* + *low variance*, and *high average gene counts* + *high variance*. 
    - A t-test assumes the variance is stable regardless of the mean.
3. **We have a low number of replicates.**
    - Calculating a reliable variance for a single gene from just four data points is statistically unstable and lacks power.

`DESeq2` is specifically designed for this. It addresses these issues by:
- Modeling the data using the correct *negative binomial* distribution.
- Normalizing counts to make them comparable across samples with different sequencing depths.
- Pooling information across all genes to get a more stable and accurate estimate of variance.

In [ ]:
# Create DESeq dataset of our data
dds = DeseqDataSet(
    counts=dex_vs_con_counts_df,
    metadata=dex_vs_con_metadata_df,
    design='~ treatment_protocol',
    refit_cooks=True,
)

# Run DESeq2 Analysis
dds.deseq2()

The log output shows the core steps `DESeq2` performs to find statistically significant gene expression changes. Here is a summary of that process:

1. **Normalization (Fitting size factors):**
    - Adjusts for differences in library size between samples. It ensures we are comparing biological changes, not technical artifacts.
2. **Model Biological Noise (Fitting dispersions):**
    - Dispersion measures a gene's natural variability across replicates. `DESeq2` creates a robust model of this noise by borrowing information across all genes, which is essential for datasets with few samples.
3. **Calculate the Signal (Fitting LFCs):**
    - With the data properly modeled, `DESeq2` calculates the Log2 Fold Change (LFC) for each gene. This is the magnitude of change between the `'Untreated'` and `'Dexamethasone'` treated samples.
4. **Find Outliers (Calculating Cook's distance)**
    - This final check identifies individual data points that might be overly influencing the results. The log replacing zero outlier genes confirms our data is clean and reliable.

Let's now run the final step to generate a DataFrame containing these statistics for each gene.

In [ ]:
deseq_stats = DeseqStats(dds, contrast=['treatment_protocol', 'Dexamethasone', 'Untreated'], quiet=True)
deseq_stats.summary()

# Get results DataFrame (log2 fold change, p-values, etc.)
results_deseq2 = deseq_stats.results_df

results_deseq2.head()

The complete differential gene analysis shows us these statistics for all the genes we compared between our `'Untreated'` and `'Dexamethosone'` groups:
- `baseMean`: The average normalized expression level of the gene across all replicates.
- `log2FoldChange`: Measures the magnitude of expression change between groups.
    - Positive values mean genes are upregulated in the Dexamethasone group compared to the Untreated group. 
    - Negative values mean genes are downregulated in the Dexamethasone group compared to the Untreated group.
- `lfcSE`: The uncertainty as standard error in the log2FoldChange estimate.
    - Smaller values indicate more precision.
- `stat`: The Wald test statistic (log2FoldChange / lfcSE) represents the **signal-to-noise ratio for the expression change**.
    - This test determines if the modeled *log2FoldChange* is significantly different from zero, making it more powerful than a t-test which simply compares raw data means.
- `pvalue`: The raw probability of observing this fold change by random chance if there were truly no difference between the groups.
- `padj`: The p-value adjusted to control for false discoveries across thousands of tests (**this is the value we will use to determine significance**).

## Visualizing Differential Gene Expression Results

We have a DataFrame with over 22,000 genes, which is too much to analyze manually. We need to visualize the results to identify the most interesting genes.

A volcano plot is the perfect tool for getting a big-picture view of our differential expression results. It plots every gene on a chart where:
- The x-axis is the `log2FoldChange` (biological effect size).
- The y-axis is the statistical significance (`-log10(padj)`).

This allows us to immediately see which genes have both a large change in expression and high statistical confidence.
We'll use the `plot_volcano()` helper function we defined at the start of this notebook to create a volcano plot for us.

In [ ]:
plot_volcano(
    results_deseq2,
    p_threshold=0.05, 
    log2fc_threshold=1,
    top_N=10
)

Now that we've visualized the results, let's create a new DataFrame containing only the genes that we consider statistically significant. To keep things simple, we'll take the top 10 most significant genes and the bottom 10 least significant genes.

In [ ]:
# Extracting the top and bottom 10 most significant genes
top_gene_counts, bot_gene_counts, top_genes, bot_genes = extract_top_bottom_genes(dex_vs_con_counts_df, results_deseq2, N=10)

display(top_gene_counts)
display(bot_gene_counts)

A heatmap is a powerful way to see the actual expression patterns of our top genes across all of our samples. It helps confirm that the changes we see are consistent within each group.

With our top 10 and bottom 10 genes, we'll visualize how many gene counts we have and compare the number of counts across all our replicates. Genes that have more counts are upregulated (orange) and genes with fewer counts are downregulated (blue).

In [ ]:
# Plot top 10 most significant genes
plot_heatmap(
    gene_counts=top_gene_counts,
    metadata=dex_vs_con_metadata_df,
    title_prefix="Top 10 Differentially Expressed Genes",
    condition_col='treatment_protocol',
    control_name='Untreated'
)

# Plot bottom 10 most significant genes
plot_heatmap(
    gene_counts=bot_gene_counts,
    metadata=dex_vs_con_metadata_df,
    title_prefix="Bottom 10 Differentially Expressed Genes",
    condition_col='treatment_protocol',
    control_name='Untreated'
)

Among the top 10 most significant genes, there is a clear increase in transcription levels compared to the control group.

In contrast, the bottom 10 genes exhibit inconsistent expression patterns across replicas; both control and treated samples display a mix of downregulated (blue) and upregulated (orange) regions, making it difficult to determine whether they are consistently up- or down-regulated, contributing to the limited significance observed in these genes.

## **Graded Exercise: (15 marks)**

**GQ*1: For Albuterol, there might be different expression patterns. Using the `sym_counts_df` select the `Untreated` and `Albuterol_Dexamethasone` subsets as `alb_dex_vs_con_counts_df`. Also, ensure the samples are correct by aquiring the subset of the `metadata_df` as `alb_dex_vs_con_metadata_df`. (3 marks)**

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# Your code here


**GQ*2: With the subset from question 1, perform DGE analysis with the `pyDESeq2` library. Using these results, generate a volcano plot with a more stringent confidence interval of 90% (p-value threshold of `0.10`) and requiring a higher baseline of differential expression (log2fc threshold of `1.2`). (6 marks)**

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# Your code here


**GQ*3: Again with the `Albuterol_Dexamethasone` subset, extract the top and bottom 10 genes and plot the heatmaps for both groups. Why is there so much contrast between the `Untreated` and `Albuterol_Dexamethasone` samples in the top genes compares to the bottom genes? (6 marks)**
> Hint: Consider what the p-value tells us about these genes

<span style="background-color: #FFD700">**Write your code below**</span>


In [ ]:
# Your code here


<span style="background-color: #FFD700">**Write your answer below**</span>

Answer here:

---

## Conclusion

In this module, we've successfully navigated a complete differential gene expression (DGE) workflow, from raw RNA-seq count data to identifying statistically significant genes. Using the `pyDESeq2` library in Python, we compared gene expression in airway smooth muscle cells between `Untreated` and `Dexamethasone` samples.

You have learned how to:
- Preprocess and clean a raw count matrix.
- Apply the robust statistical methods of `DESeq2`, which are specifically designed for bulk RNA-seq data.
- Interpret key metrics like `log2FoldChange` and adjusted p-values (`padj`).
- Visualize the results effectively using volcano plots and heatmaps to highlight the characteristics of the most/least impactful genes.

This foundational analysis is a critical first step in transcriptomics, paving the way for downstream functional analysis to understand the biological impact of these expression changes. You are now equipped with the fundamental skills to analyze bulk RNA-seq data and uncover meaningful biological insights.